In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
file_path = "/Volumes/Engineering/Database/HMCorp/java/train.jsonl"
hmcorp = pd.read_json(file_path, lines=True)

In [5]:
df = pd.read_csv("../data/hmcorp_xml.csv")

print("Shape of output:", df.shape)

print(df.columns)


Shape of output: (355736, 5)
Index(['id', 'code', 'label', 'language', 'xml'], dtype='str')


In [6]:
df.head()

,id,code,label,language,xml
0,gj235730,public Object readValue(Object value) {\n /...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
1,gj235730,private OptionKindAndValue readKindAndValue() ...,0,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
2,gj164536,public <U> CompletableFuture<U> thenComposeAsy...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
3,gj164536,public <TContinuationResult> Task<TContinuatio...,0,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
4,gj096930,public String getUserAgent() {\n String jav...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."


In [7]:
unique_ids = df["id"].unique()
print(unique_ids)

<StringArray>
['gj235730', 'gj164536', 'gj096930', 'gj008079', 'gj166166', 'gj178131',
 'gj097050', 'gj095207', 'gj160320', 'gj017145',
 ...
 'gj058951', 'gj181773', 'gj196606', 'gj107861', 'gj136229', 'gj106691',
 'gj163801', 'gj160435', 'gj096701', 'gj269528']
Length: 177868, dtype: str


In [8]:
import re

_FILENAME_ATTR = re.compile(r'\s+filename="[^"]*"')


def strip_xml_leak(xml: str) -> str:
    return _FILENAME_ATTR.sub("", xml)


In [9]:
# Allow the column width to expand infinitely
pd.set_option('display.max_colwidth', None)

# Print the full string
print(df["xml"][2])

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<unit xmlns="http://www.srcML.org/srcML/src" revision="1.0.0" language="Java" filename="temp/gj164536_human.java"><function><type><specifier>public</specifier> <parameter_list type="generic">&lt;<parameter><name>U</name></parameter>&gt;</parameter_list> <name><name>CompletableFuture</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></type> <name>thenComposeAsync</name><parameter_list>(<parameter><decl><type><name><name>Function</name><argument_list type="generic">&lt;<argument><name>?</name> <super>super <name>T</name></super></argument>, <argument><name>?</name> <extends>extends <name><name>CompletionStage</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></extends></argument>&gt;</argument_list></name></type> <name>fn</name></decl></parameter>)</parameter_list> <block>{<block_content>
    <return>return <expr><name><name>Compl

In [10]:
df["xml"] = df["xml"].apply(strip_xml_leak)

In [11]:
# Print the full string
print(df["xml"][2])

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<unit xmlns="http://www.srcML.org/srcML/src" revision="1.0.0" language="Java"><function><type><specifier>public</specifier> <parameter_list type="generic">&lt;<parameter><name>U</name></parameter>&gt;</parameter_list> <name><name>CompletableFuture</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></type> <name>thenComposeAsync</name><parameter_list>(<parameter><decl><type><name><name>Function</name><argument_list type="generic">&lt;<argument><name>?</name> <super>super <name>T</name></super></argument>, <argument><name>?</name> <extends>extends <name><name>CompletionStage</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></extends></argument>&gt;</argument_list></name></type> <name>fn</name></decl></parameter>)</parameter_list> <block>{<block_content>
    <return>return <expr><name><name>CompletableFuture</name><operator>.</oper

In [12]:
def findKeywords(xml):
    xml = xml.lower()
    if "human" in xml:
        return "human"
    # if "ai" in xml:
    #     return "ai"
    return None

df["keyword"] = df["xml"].apply(findKeywords)

count = df["keyword"].notna().sum()
print(len(df))
print(count)


355736
135


In [13]:
flagged_df = df[df["keyword"].notna()]

In [14]:
df = df[df["keyword"].isna()].copy()
df = df.drop(index=flagged_df.index).copy()

KeyError: '[3269, 6781, 7381, 18125, 20231, 23697, 25556, 32343, 32478, 32505, 37156, 39575, 41643, 50858, 55655, 56982, 61605, 68533, 72736, 72838, 73027, 73494, 78174, 78175, 82852, 85271, 86815, 87245, 88485, 90306, 96406, 96407, 100220, 102998, 107099, 107624, 107777, 112561, 114810, 114811, 115100, 115101, 123267, 124032, 124033, 125098, 128395, 136525, 139507, 143509, 148678, 148679, 150697, 157360, 157361, 158015, 158259, 159052, 159053, 161242, 161721, 166833, 169348, 169456, 171439, 173103, 177671, 179884, 183419, 187105, 188231, 190688, 192019, 195740, 197976, 200889, 205242, 208843, 212994, 215599, 216680, 216681, 222062, 222415, 228123, 230293, 231258, 231259, 233610, 233611, 234313, 235963, 239913, 243403, 245288, 245289, 248280, 248281, 249218, 249219, 249896, 250444, 253069, 254567, 258377, 265437, 269367, 269605, 271784, 272310, 273007, 280714, 280715, 281095, 283499, 284018, 284153, 285537, 285594, 286300, 291206, 291887, 294595, 299466, 301292, 302533, 303838, 305885,

In [ ]:
print(df["keyword"].notna().sum())
print(df.shape)

0
(355601, 6)


In [ ]:
df.drop(columns=["keyword"], inplace=True)
df.to_csv("../data/hmcorp_xml_2.csv", index=False)

In [ ]:
df2 = pd.read_csv("../data/hmcorp_xml_2.csv")

In [ ]:
df2.shape

(355601, 5)

In [ ]:
def findKeywords(xml):
    xml = xml.lower()
    if "human" in xml:
        return "human"
    # if "ai" in xml:
    #     return "ai"
    return None

df2["keyword"] = df2["xml"].apply(findKeywords)
count = df2["keyword"].notna().sum()
print(count)

0


In [ ]:
3,55,736

In [15]:
train_df = pd.read_csv("../split/train.csv")
test_df = pd.read_csv("../split/test.csv")

In [18]:
train_df.shape
print(train_df.columns)

Index(['id', 'code', 'label', 'language', 'xml'], dtype='str')


In [20]:
overlap = train_df[train_df["id"].isin(test_df["id"])]

print(f"Overlapping ids: {overlap['id'].nunique()}")
print(overlap["id"].unique())

Overlapping ids: 0
<StringArray>
[]
Length: 0, dtype: str
